# Once Upon a Mirror, notebook 3: Estimate then exploit

In section 3 of the game, the mirror first showed each product once, then drew a card from the hat on each of n
mornings, as Random does. Then it picked the product with the highest estimate and showed it on every remaining
morning. Here you write that strategy in Python, play it on the game's race season, and try different n on the
two collections of the challenge round.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A folded cell with a title holds plumbing: run it, and open it
only if you are curious. The 💡 hints are folded cells too: double-click one to open it, and open the next one only
if you are still stuck. After each task, a check cell tells you whether your code works. It takes about 25
minutes.

The products, the customers and their probabilities are invented for the game. Start with the three folded cells
below: the mirror's kit, the checks, and the two mirrors of notebook 2.

In [ ]:
#@title The mirror's kit (run me first) { display-mode: "form" }
# The mirror's kit. Run this cell first; you do not need to read it.
SEASON = 100            # mornings in a season
RACE = 103993           # the seed of the race season: the season of every comparison in the game
NAMES = ['Robot vacuum', 'Rope ladder', 'Leaf blower', 'Memory-foam mattress', 'Satnav']
EMOJI = ['🤖', '🪜', '🍃', '🛏️', '🧭']
_KIT_STREAMS = {"random": 1, "estimate then exploit": 2, "epsilon-greedy": 3, "ucb": 4}

def random_numbers(seed):
    """A stream of random numbers between 0 and 1: next_number = random_numbers(7), then next_number() gives the
    first number, next_number() again the second, and so on. The same seed gives the same numbers as in the game."""
    state = seed & 0xFFFFFFFF

    def next_number():
        nonlocal state
        state = (state + 0x6D2B79F5) & 0xFFFFFFFF
        t = ((state ^ (state >> 15)) * (1 | state)) & 0xFFFFFFFF
        t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296

    return next_number


def strategy_numbers(seed, strategy):
    """A strategy's own stream of random numbers in the season with this seed, as in the game. strategy is one of
    "random", "estimate then exploit", "epsilon-greedy" and "ucb"."""
    return random_numbers(10 * seed + _KIT_STREAMS[strategy])


def season_moods(seed):
    """The customer's mood on each morning of the season with this seed: SEASON numbers between 0 and 1."""
    next_number = random_numbers(seed)
    moods = []
    for _ in range(SEASON):
        moods.append(next_number())
    return moods


def tap(p, mood):
    """1 if the customer taps the product shown, else 0. She taps when her mood is below p, which happens on a
    share p of the mornings. Every mirror of a season meets the same moods, so the comparison is fair."""
    return 1 if mood < p else 0


def _kit_is_board(rows):
    if not isinstance(rows, (list, tuple)) or len(rows) == 0:
        return False
    for row in rows:
        if not isinstance(row, (list, tuple)):
            return False
        for x in row:
            if x not in (0, 1):
                return False
    return True


def _kit_number(x):
    """At most two decimals, as in the game: 0.6, 0.67, 1, 0."""
    text = "%.2f" % x
    return text.rstrip("0").rstrip(".") if "." in text else text


def rate(rows):
    """The rate of a season: taps / mornings over the whole board."""
    if not _kit_is_board(rows):
        print("⏳ rate needs a board: one list of 1s and 0s per product. It got %s." % _kit_short(rows))
        return None
    taps = sum(sum(row) for row in rows)
    mornings = sum(len(row) for row in rows)
    return taps / mornings if mornings else 0


def _kit_short(x):
    text = repr(x)
    return text if len(text) <= 60 else text[:57] + "..."


def show_board(rows, symbols=None, highlight=None, width=30, estimates=True):
    """Print the board as in the game: the estimate p-hat, the product, then one dot per showing, 🟠 a tap and
    ⚪ no tap. highlight: a row (or a list of rows) to mark with ⭐. A row longer than width is cut and its
    remaining dots are counted."""
    if not _kit_is_board(rows):
        print("⏳ No board to show yet: show_board needs one list of 1s and 0s per product. It got %s." % _kit_short(rows))
        return
    symbols = symbols or EMOJI
    marked = highlight if isinstance(highlight, (list, tuple)) else ([] if highlight is None else [highlight])
    if estimates:
        print("    p\u0302")
    for k in range(len(rows)):
        row = rows[k]
        est = ("?" if not row else _kit_number(sum(row) / len(row))) if estimates else ""
        dots = "".join("🟠" if x else "⚪" for x in row[:width])
        if len(row) > width:
            rest = row[width:]
            dots += " ... and %d more (%d 🟠)" % (len(rest), sum(rest))
        name = symbols[k] if k < len(symbols) else "#%d" % k
        print("%5s  %s  %s%s" % (est, name, dots, "  ⭐" if k in marked else ""))

def estimate(row):
    """The estimate of a product: the average of its 1s and 0s (taps / showings); None when it was never shown."""
    if len(row) == 0:
        return None
    return sum(row) / len(row)


def estimates(rows):
    """The estimate of every product on the board, in the order of the rows."""
    return [estimate(row) for row in rows]

def pick_highest(values, next_number):
    """The row with the highest value. When several rows share the highest value, one of them at random, with a
    number from next_number: the rule of the game, so the same stream gives the same choice. A number is drawn only
    when there is a tie."""
    best = max(values)
    tied = [k for k in range(len(values)) if abs(values[k] - best) <= 1e-9]
    if len(tied) == 1:
        return tied[0]
    return tied[int(next_number() * len(tied))]

In [ ]:
#@title The checks of notebook 3 (run me too) { display-mode: "form" }
# The checks of notebook 3, and play, which runs your mirror calmly. You do not need to read this cell.
# A check prints one short message and never stops the notebook.

# Each test is a season (its probabilities, its seed, its mornings and n) and what the game's estimate then exploit
# does on it, morning by morning: "shown" holds the row shown on each morning and "dots" her answer, 1 a tap and 0 no
# tap. "test" counts the mornings of the initialization and the test, "tied" holds the rows that share the highest
# estimate after it, and "winner" is the row the game picks (-1 when the test fills the season). "slips" holds the
# rows that a few usual slips show on the same season, morning by morning, to name them.
_N3_TESTS = {'initialize_and_test': [{'about': "the race season at Cinderella's, n = 20", 'call': 'initialize_and_test(cinderella, season_moods(RACE), strategy_numbers(RACE, "estimate then exploit"), 20)', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 103993, 'mornings': 100, 'n': 20, 'test': 25, 'shown': '0123412210333344423043213000000000000000000000000000000000000000000000000000000000000000000000000000', 'dots': '0001000001001100000110000001111101001110000011010011001011110000000111111001111111111100110011100000', 'tied': [0], 'winner': 0, 'slips': {'short': '0123412210333344423043210000000000000000000000000000000000000000000000000000000000000000000000000000'}}, {'about': "the race season at Rapunzel's, n = 10", 'call': 'initialize_and_test(rapunzel, season_moods(RACE), strategy_numbers(RACE, "estimate then exploit"), 10)', 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 103993, 'mornings': 100, 'n': 10, 'test': 15, 'shown': '0123412210333341111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'dots': '0101010010000001111111111101111101101111001011110111011011111001010111111001111111111101111011110010', 'tied': [1], 'winner': 1, 'slips': {'short': '0123412210333311111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'long': '0123412210333344111111111111111111111111111111111111111111111111111111111111111111111111111111111111'}}, {'about': 'the collection Ready for the ball, from the challenge round, n = 0', 'call': 'initialize_and_test([0.6, 0.9, 0.6, 0.95, 0.6], season_moods(1), strategy_numbers(1, "estimate then exploit"), 0)', 'p': [0.6, 0.9, 0.6, 0.95, 0.6], 'names': ['silk gloves', 'tiara', 'evening bag', 'ball gown', 'lipstick'], 'seed': 1, 'mornings': 100, 'n': 0, 'test': 5, 'shown': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'dots': '0110011110111111111111111111111111101111111111111110111100101011111101110101111111111111111111111111', 'tied': [1, 2], 'winner': 1, 'slips': {'every': '0123411211111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111'}}, {'about': 'the collection Long hair, from the challenge round, n = 95', 'call': 'initialize_and_test([0.02, 0.02, 0.95, 0.02, 0.02], season_moods(2), strategy_numbers(2, "estimate then exploit"), 95)', 'p': [0.02, 0.02, 0.95, 0.02, 0.02], 'names': ['scissors', 'cap', 'ribbon', 'razor', 'haircut voucher'], 'seed': 2, 'mornings': 100, 'n': 95, 'test': 100, 'shown': '0123431111423300002021233303200320140101101100203300214023242012323401300132410032111040204124132214', 'dots': '0010000000010000000010101000000010000010000000100000100010101001011000000001000001000000100010001100', 'tied': [], 'winner': -1, 'slips': {'short': '0123431111423300002021233303200320140101101100203300214023242012323401300132410032111040204124132212'}}, {'about': "another season at Cinderella's, n = 50", 'call': 'initialize_and_test(cinderella, season_moods(3), strategy_numbers(3, "estimate then exploit"), 50)', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 3, 'mornings': 100, 'n': 50, 'test': 55, 'shown': '0123413142202341013211132120441124340244343001233033433000000000000000000000000000000000000000000000', 'dots': '0101000100110000101000010000001001100000001110000100010010000001110111111101000011100000010001001110', 'tied': [0], 'winner': 0, 'slips': {'every': '0123413142202341013211132120441124340244343001233033433000000300000000000000000000000000000000000000', 'short': '0123413142202341013211132120441124340244343001233033430000000000000000000000000000000000000000000000', 'long': '0123413142202341013211132120441124340244343001233033433200000000000000000000000000000000000000000000'}}, {'about': 'the collection Ready for the ball, from the challenge round, n = 3', 'call': 'initialize_and_test([0.6, 0.9, 0.6, 0.95, 0.6], season_moods(4), strategy_numbers(4, "estimate then exploit"), 3)', 'p': [0.6, 0.9, 0.6, 0.95, 0.6], 'names': ['silk gloves', 'tiara', 'evening bag', 'ball gown', 'lipstick'], 'seed': 4, 'mornings': 100, 'n': 3, 'test': 8, 'shown': '0123432433333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333', 'dots': '0111111011111111111101111111111111111111111011111111011111011111111111110111110110101111111111111111', 'tied': [1, 2, 3], 'winner': 3, 'slips': {'every': '0123432431213311331133333333333333333333333333333333333333333333333333333333333333333333333333333333', 'first': '0123432411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'short': '0123432444444444444444444444444444444444444444444444444444444444444444444444444444444444444444444444', 'long': '0123432431111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111'}}], 'estimate_then_exploit': [{'about': "the race season at Cinderella's, n = 20", 'call': 'estimate_then_exploit(cinderella, season_moods(RACE), strategy_numbers(RACE, "estimate then exploit"), 20)', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 103993, 'mornings': 100, 'n': 20, 'test': 25, 'shown': '0123412210333344423043213000000000000000000000000000000000000000000000000000000000000000000000000000', 'dots': '0001000001001100000110000001111101001110000011010011001011110000000111111001111111111100110011100000', 'tied': [0], 'winner': 0, 'slips': {'short': '0123412210333344423043210000000000000000000000000000000000000000000000000000000000000000000000000000'}}, {'about': "another season at Cinderella's, n = 17", 'call': 'estimate_then_exploit(cinderella, season_moods(16), strategy_numbers(16, "estimate then exploit"), 17)', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 16, 'mornings': 100, 'n': 17, 'test': 22, 'shown': '0123444120142012043121333333333333333333333333333333333333333333333333333333333333333333333333333333', 'dots': '0101010010010100100100000000110100000001000011001111010110100111100000100100110010001100010011101100', 'tied': [0, 3], 'winner': 3, 'slips': {'every': '0123444120142012043121300000000000000000000000000000000000000000000000000000000000000000000000000000', 'first': '0123444120142012043121000000000000000000000000000000000000000000000000000000000000000000000000000000', 'short': '0123444120142012043120000000000000000000000000000000000000000000000000000000000000000000000000000000', 'long': '0123444120142012043121200000000000000000000000000000000000000000000000000000000000000000000000000000'}}, {'about': "the race season at Cinderella's, n = 6", 'call': 'estimate_then_exploit(cinderella, season_moods(RACE), strategy_numbers(RACE, "estimate then exploit"), 6)', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 103993, 'mornings': 100, 'n': 6, 'test': 11, 'shown': '0123412210333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333', 'dots': '0001000001001100110010000001101101001010000001000001001010110000000010101001111101101000110001100000', 'tied': [0, 3], 'winner': 3, 'slips': {'every': '0123412210330000000000000000000000000000000000000000000000000000000000000000000000000000000000000000', 'first': '0123412210300000000000000000000000000000000000000000000000000000000000000000000000000000000000000000', 'long': '0123412210330000000000000000000000000000000000000000000000000000000000000000000000000000000000000000'}}, {'about': "the race season at Rapunzel's, n = 10", 'call': 'estimate_then_exploit(rapunzel, season_moods(RACE), strategy_numbers(RACE, "estimate then exploit"), 10)', 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 103993, 'mornings': 100, 'n': 10, 'test': 15, 'shown': '0123412210333341111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'dots': '0101010010000001111111111101111101101111001011110111011011111001010111111001111111111101111011110010', 'tied': [1], 'winner': 1, 'slips': {'short': '0123412210333311111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'long': '0123412210333344111111111111111111111111111111111111111111111111111111111111111111111111111111111111'}}, {'about': 'the collection Ready for the ball, from the challenge round, n = 0', 'call': 'estimate_then_exploit([0.6, 0.9, 0.6, 0.95, 0.6], season_moods(3), strategy_numbers(3, "estimate then exploit"), 0)', 'p': [0.6, 0.9, 0.6, 0.95, 0.6], 'names': ['silk gloves', 'tiara', 'evening bag', 'ball gown', 'lipstick'], 'seed': 3, 'mornings': 100, 'n': 0, 'test': 5, 'shown': '0123422222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222', 'dots': '0111011111110100101010110100101101110000111110111100010010000001110111111101000011100000010001001110', 'tied': [1, 2, 3], 'winner': 2, 'slips': {'every': '0123423132212331113311133131331113331133333111133133333313111133333333333333311111111113133311113333', 'first': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'long': '0123413333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333'}}, {'about': 'the collection Ready for the ball, from the challenge round, n = 25', 'call': 'estimate_then_exploit([0.6, 0.9, 0.6, 0.95, 0.6], season_moods(2), strategy_numbers(2, "estimate then exploit"), 25)', 'p': [0.6, 0.9, 0.6, 0.95, 0.6], 'names': ['silk gloves', 'tiara', 'evening bag', 'ball gown', 'lipstick'], 'seed': 2, 'mornings': 100, 'n': 25, 'test': 30, 'shown': '0123431111423300002021233303201111111111111111111111111111111111111111111111111111111111111111111111', 'dots': '0111011111011111010101111111011111110110101111101101101111111100111011110111110111111111111110111111', 'tied': [1, 3], 'winner': 1, 'slips': {'every': '0123431111423300002021233303201311131333331111111113331111111111111133333333333333333333333333333333', 'short': '0123431111423300002021233303211111111111111111111111111111111111111111111111111111111111111111111111', 'long': '0123431111423300002021233303200333333333333333333333333333333333333333333333333333333333333333333333'}}, {'about': 'the collection Long hair, from the challenge round, n = 3', 'call': 'estimate_then_exploit([0.02, 0.02, 0.95, 0.02, 0.02], season_moods(6), strategy_numbers(6, "estimate then exploit"), 3)', 'p': [0.02, 0.02, 0.95, 0.02, 0.02], 'names': ['scissors', 'cap', 'ribbon', 'razor', 'haircut voucher'], 'seed': 6, 'mornings': 100, 'n': 3, 'test': 8, 'shown': '0123430011111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'dots': '0110001000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000', 'tied': [1, 2], 'winner': 1, 'slips': {'every': '0123430012222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222', 'short': '0123430111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'long': '0123430021111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111'}}, {'about': 'the collection Long hair, from the challenge round, n = 95', 'call': 'estimate_then_exploit([0.02, 0.02, 0.95, 0.02, 0.02], season_moods(7), strategy_numbers(7, "estimate then exploit"), 95)', 'p': [0.02, 0.02, 0.95, 0.02, 0.02], 'names': ['scissors', 'cap', 'ribbon', 'razor', 'haircut voucher'], 'seed': 7, 'mornings': 100, 'n': 95, 'test': 100, 'shown': '0123410100144412103022332031301034410201140333433414333343233140141440110034404331303401410312241333', 'dots': '1000000000000001000011001000000000000100000000000000000000100000000000000000010000000000000011100100', 'tied': [], 'winner': -1, 'slips': {'short': '0123410100144412103022332031301034410201140333433414333343233140141440110034404331303401410312241332'}}, {'about': "another season at Cinderella's, n = 50", 'call': 'estimate_then_exploit(cinderella, season_moods(8), strategy_numbers(8, "estimate then exploit"), 50)', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 8, 'mornings': 100, 'n': 50, 'test': 55, 'shown': '0123434232141132423131400424103322013131140222210214023000000000000000000000000000000000000000000000', 'dots': '1000000001010010011100001000011000000000011000100000100111100110001111101011000100111110111011010011', 'tied': [0], 'winner': 0, 'slips': {'short': '0123434232141132423131400424103322013131140222210214020000000000000000000000000000000000000000000000', 'long': '0123434232141132423131400424103322013131140222210214023400000000000000000000000000000000000000000000'}}]}
_N3_SUCCESS = {'initialize_and_test': "initialize_and_test works on 6 seasons, morning by morning as in the game, n from 0 to 95. On Cinderella's race season with n = 20 it gives the game's board after the test: the vacuum 2 taps in 3 showings, the mattress 3 in 8, the satnav 1 in 5, and no tap for the ladder and the blower.", 'estimate_then_exploit': "estimate_then_exploit works on 9 seasons, morning by morning as in the game, n from 0 to 95. On Cinderella's race season with n = 20 it picks the vacuum, 2/3 against the mattress's 3/8, and gets 47 taps in 100 mornings, a rate of 0.47: the game's numbers."}
_N3_TASKS = {"initialize_and_test": "task A", "estimate_then_exploit": "task B"}
_N3_NEEDS = {"initialize_and_test": [], "estimate_then_exploit": ["initialize_and_test"]}
_N3_GAPS = {}
_N3_WORKING = {}
_N3_LOG = []


def _n3_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = repr(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 70:
        text = text[:67] + "..."
    return text


def _n3_count(n, word):
    """1 dot, 3 dots"""
    if n == 1:
        return "1 " + word
    return "%d %ss" % (n, word)


def _n3_ordinal(n):
    """1st, 2nd, 3rd, 4th, 11th, 21st"""
    if n % 100 in (11, 12, 13):
        return "%dth" % n
    return "%d%s" % (n, {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th"))


def _n3_morning(t):
    """morning 12 (the 13th of the season): Python counts the mornings from 0."""
    return "morning %d (the %s of the season)" % (t, _n3_ordinal(t + 1))


def _n3_and(items):
    """a, b and c"""
    items = [str(x) for x in items]
    if len(items) == 1:
        return items[0]
    return ", ".join(items[:-1]) + " and " + items[-1]


def _n3_product(test, k):
    """the rope ladder (row 1, p = 0.2)"""
    if not isinstance(k, int) or k < 0 or k >= len(test["p"]):
        return "row %s" % _n3_short(k)
    return "the %s (row %d, p = %s)" % (test["names"][k], k, test["p"][k])


def _n3_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    code = getattr(fn, "__code__", None)
    if code is None:
        return False
    if code in _N3_GAPS:
        return _N3_GAPS[code]
    # A gap such as "test = ..." stays in the compiled function; a gap on a line of its own is only visible in the
    # source, which Jupyter and Colab keep for every cell
    gap = Ellipsis in code.co_consts
    if not gap:
        import inspect
        try:
            for line in inspect.getsource(fn).splitlines():
                if "..." in line.split("#")[0]:
                    gap = True
        except Exception:
            pass
    _N3_GAPS[code] = gap
    return gap


def _n3_is_board(value, rows=None):
    """True when value is a board: a list of lists of 1s and 0s (and rows lists, when rows is given)."""
    if not isinstance(value, list) or len(value) == 0:
        return False
    if rows is not None and len(value) != rows:
        return False
    for row in value:
        if not isinstance(row, list):
            return False
        for x in row:
            if not isinstance(x, (bool, int, float)) or x not in (0, 1):
                return False
    return True


def _n3_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


# The check follows your mirror morning by morning. It hands your function moods and probabilities that write down
# every comparison of a mood with a probability, which is what tap does, and a next_number that writes down every
# number it gives. So it knows which product your mirror showed on which morning, and which numbers it drew, whatever
# the shape of your code.

def _n3_note(prob, mood, swapped):
    """One comparison: ("tap", row, morning, swapped, value) for a mood and a probability (row None and the value
    itself when it is not one of p), ("p", ...) for two probabilities."""
    if isinstance(mood, _n3_Mood):
        row = None
        if isinstance(prob, _n3_Prob):
            row = prob.row
        _N3_LOG.append(("tap", row, mood.morning, swapped, prob))
    elif isinstance(prob, _n3_Prob) and isinstance(mood, _n3_Prob):
        _N3_LOG.append(("p", None, None, False, None))


class _n3_Mood(float):
    """A mood that knows its morning: mood < p is her answer, the right way round."""

    def __lt__(self, other):
        _n3_note(other, self, False)
        return float.__lt__(self, other)

    def __le__(self, other):
        _n3_note(other, self, False)
        return float.__le__(self, other)

    def __gt__(self, other):
        _n3_note(other, self, True)
        return float.__gt__(self, other)

    def __ge__(self, other):
        _n3_note(other, self, True)
        return float.__ge__(self, other)


class _n3_Prob(float):
    """A probability that knows its row: p > mood is her answer, the right way round."""

    def __gt__(self, other):
        _n3_note(self, other, False)
        return float.__gt__(self, other)

    def __ge__(self, other):
        _n3_note(self, other, False)
        return float.__ge__(self, other)

    def __lt__(self, other):
        _n3_note(self, other, True)
        return float.__lt__(self, other)

    def __le__(self, other):
        _n3_note(self, other, True)
        return float.__le__(self, other)


def _n3_inputs(name, test):
    """Fresh inputs for a test, [p, moods, next_number, n]: moods and probabilities that write down each comparison,
    and a next_number that writes down each number it gives."""
    moods = season_moods(test["seed"])[:test["mornings"]]
    p = []
    for k in range(len(test["p"])):
        x = _n3_Prob(test["p"][k])
        x.row = k
        p.append(x)
    marked = []
    for t in range(len(moods)):
        x = _n3_Mood(moods[t])
        x.morning = t
        marked.append(x)
    numbers = strategy_numbers(test["seed"], "estimate then exploit")

    def next_number():
        value = numbers()
        _N3_LOG.append(("draw", None, None, False, value))
        return value
    return [p, marked, next_number, test["n"]]


def _n3_run(fn, inputs):
    """Run fn on the inputs with its prints silenced and read what it did: answer, error (None when the call went
    through) and line (the line of your code where it happened); events, one (row, morning, swapped, value) per
    answer of the customer, in order (value is what was compared with the mood); draws, one (answers before it,
    value) per number drawn from next_number; p_read, True when it compared two probabilities. Never raises."""
    import contextlib, io, traceback
    del _N3_LOG[:]
    run = {"answer": None, "error": None, "line": None}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            run["answer"] = fn(*inputs)
    except Exception as error:
        # The check's moods and probabilities are floats that write down each comparison: name them float
        run["error"] = ("%s: %s" % (type(error).__name__, error)).replace("_n3_Mood", "float").replace("_n3_Prob",
                                                                                                    "float")
        try:
            skip = (_n3_run.__code__.co_filename, tap.__code__.co_filename)
            for frame in traceback.extract_tb(error.__traceback__):
                if frame.filename not in skip and frame.line:
                    run["line"] = frame.line.strip()
        except Exception:
            run["line"] = None
    events, draws, p_read = [], [], False
    for kind, row, morning, swapped, value in _N3_LOG:
        if kind == "tap":
            events.append((row, morning, swapped, value))
        elif kind == "draw":
            draws.append((len(events), value))
        else:
            p_read = True
    del _N3_LOG[:]
    run["events"], run["draws"], run["p_read"] = events, draws, p_read
    return run


def _n3_expected(name, test):
    """The game's board after the test (task A) or after the whole season (task B)."""
    upto = test["mornings"]
    if name == "initialize_and_test":
        upto = test["test"]
    rows = []
    for k in range(len(test["p"])):
        rows.append([])
    for t in range(upto):
        rows[int(test["shown"][t])].append(int(test["dots"][t]))
    return rows


def _n3_first_failure(name, fn):
    """The first test that fn fails, with what it did, or None when fn passes them all."""
    for test in _N3_TESTS[name]:
        run = _n3_run(fn, _n3_inputs(name, test))
        expected = _n3_expected(name, test)
        same = False
        if run["error"] is None:
            try:
                same = _n3_is_board(run["answer"]) and run["answer"] == expected
            except Exception:
                same = False
        if not same:
            run["test"] = test
            run["expected"] = expected
            return run
    return None


def _n3_works(name):
    """True when the function called name exists, is finished and passes its tests. Remembered for each version of
    the function, so play can ask on every season."""
    fn = globals().get(name)
    code = getattr(fn, "__code__", None)
    if not callable(fn) or code is None or _n3_has_gap(fn):
        return False
    if code not in _N3_WORKING:
        _N3_WORKING[code] = _n3_first_failure(name, fn) is None
    return _N3_WORKING[code]


def _n3_index(name, test, morning):
    """The row of the line with tap: product during the initialization and the test, winner after it."""
    if name == "initialize_and_test" or morning < test["test"]:
        return "product"
    return "winner"


def _n3_drawn(run, j):
    """How many numbers your mirror drew from next_number before her answer on morning j."""
    count = 0
    for before, value in run["draws"]:
        if before <= j:
            count = count + 1
    return count


def _n3_number_at(run, j):
    """The last number your mirror drew right before her answer on morning j, or None."""
    found = None
    for before, value in run["draws"]:
        if before == j:
            found = value
    return found


def _n3_game_drawn(test, j):
    """How many numbers the game's mirror drew before her answer on morning j: one card on each morning of the hat,
    then one for a tie after the test."""
    k, length = len(test["p"]), test["test"]
    if j < k:
        return 0
    if j < length:
        return j - k + 1
    if len(test["tied"]) > 1:
        return test["n"] + 1
    return test["n"]


# The hints

def _n3_explain_error(name, run):
    """The hint for a function that stopped with an error."""
    error = run["error"]
    text = (run["line"] or "").replace(" ", "")
    test = run["test"]
    last = len(test["p"]) - 1
    if "positional argument" in error:
        first = {"initialize_and_test": "def initialize_and_test(p, moods, next_number, n):",
                 "estimate_then_exploit": "def estimate_then_exploit(p, moods, next_number, n):"}[name]
        return "keep the first line of the function as it was given: " + first
    if error.startswith("NameError"):
        if "initialize_and_test" in error:
            return "initialize_and_test is your function from task A: run its cell, then this one."
        if "'rows'" in error and name == "estimate_then_exploit":
            return ("pick the winner after the test: rows = initialize_and_test(p, moods, next_number, n) comes "
                    "first, then winner = pick_highest(estimates(rows), next_number). The cards of the hat come out "
                    "of next_number first, and a tie takes the next number after them.")
        if "random" in error or "randint" in error or "choice" in error:
            return ("the cards of the hat come from next_number, the mirror's own stream, as in the game: product = "
                    "int(next_number() * len(p)).")
        known = "p, moods, next_number, n, rows, test, morning, product, dot and tap"
        if name == "estimate_then_exploit":
            known = ("p, moods, next_number, n, rows, test, winner, morning, dot, tap, initialize_and_test, "
                     "estimates and pick_highest")
        return "Python does not know a name in your function: check its spelling. Inside it you can use " + known + "."
    if error.startswith("ValueError") and "max()" in error:
        return ("pick the winner after the test, from the board that initialize_and_test gives: rows = "
                "initialize_and_test(p, moods, next_number, n), then winner = pick_highest(estimates(rows), "
                "next_number). The cards of the hat come out of next_number first, and a tie takes the next number "
                "after them.")
    if error.startswith("IndexError"):
        if "moods[" in text and len(run["events"]) >= test["mornings"]:
            if name == "initialize_and_test":
                return ("your test went past the last morning of the season. The initialization takes len(p) mornings "
                        "and the hat n more: test = len(p) + n, %d mornings for n = %d." % (test["test"], test["n"]))
            return ("a morning went past the end of the season. The remaining mornings are range(test, len(moods)), "
                    "from the first morning after the test to the last one.")
        if "winner" in text:
            return "winner is a row from 0 to %d: the row that pick_highest(estimates(rows), next_number) gives." % last
        if "p[" in text or "rows[" in text:
            return ("product is a row from 0 to %d. In the initialization product = morning, and on a morning of the "
                    "hat int(next_number() * len(p)) gives a row from 0 to %d." % (last, last))
        return ("an index went past the end of a list. Python counts from 0: the rows are 0 to %d, the mornings 0 to "
                "%d." % (last, test["mornings"] - 1))
    if error.startswith("TypeError"):
        if "NoneType" in error and ("pick_highest" in text or "estimates" in text or "max(" in text):
            return ("estimates gives None for a row with no dots yet. Pick the winner after the test, from the board "
                    "that initialize_and_test gives: winner = pick_highest(estimates(rows), next_number).")
        if "indices must be integers" in error and "float" in error:
            if "winner" in text:
                return ("winner is a row number, such as 0 or 3. pick_highest(estimates(rows), next_number) gives the "
                        "row with the highest estimate; max gives the estimate itself.")
            return ("a row number is a whole number: int(next_number() * len(p)) turns the number from the hat into "
                    "one.")
        if "'function'" in error and "operand" in error:
            return "next_number is the stream of the hat: next_number() with brackets gives its next number."
        if "not callable" in error and name == "estimate_then_exploit":
            return ("give pick_highest the stream itself, with no brackets after next_number: "
                    "pick_highest(estimates(rows), next_number).")
        if "not supported between" in error:
            return ("tap compares two numbers: the probability of the product shown and the mood of the morning, "
                    "tap(p[%s], moods[morning])." % _n3_index(name, test, test["mornings"]))
        if "range" in text or "cannot be interpreted as an integer" in error:
            return "test is a whole number of mornings: test = len(p) + n."
    if error.startswith("AttributeError") and "append" in error:
        return ("append works on a list. rows[%s] is the row of the product shown, a list, and rows[%s].append(dot) "
                "adds the dot at its end." % (_n3_index(name, test, 0), _n3_index(name, test, 0)))
    if name == "initialize_and_test":
        return ("initialize with product = morning, then on each morning of the hat product = int(next_number() * "
                "len(p)), then her answer and its dot, as given.")
    return ("rows = initialize_and_test(p, moods, next_number, n), then winner = pick_highest(estimates(rows), "
            "next_number), then the loop over the remaining mornings.")


def _n3_shape(name, run):
    """(what, hint) when fn did not give a board with one row per product, or None."""
    test, answer = run["test"], run["answer"]
    rows = len(test["p"])
    index = "product"
    if name == "estimate_then_exploit":
        index = "winner"
    if answer is None:
        return ("your function gave no answer (None)", "end the function with return rows, after the loop.")
    if not isinstance(answer, list):
        return ("your function gave %s, and a board is a list of rows" % _n3_short(answer),
                "return rows, the board: one list per product.")
    if len(answer) == 0:
        return ("your function gave an empty board, []",
                "the board starts with one empty row per product: rows.append([]) in the loop over the products.")
    for row in answer:
        if not isinstance(row, list):
            return ("your function gave %s, a board with numbers next to its rows" % _n3_short(answer),
                    "add each dot to the row of the product shown: rows[%s].append(dot). rows.append(dot) puts it "
                    "next to the rows." % index)
    if len(answer) != rows:
        return ("your function gave a board of %s for %s" % (_n3_count(len(answer), "row"), _n3_count(rows, "product")),
                "one row per product: the board starts with one empty row for each probability in p.")
    for r in range(rows):
        for x in answer[r]:
            if not isinstance(x, (bool, int, float)) or x not in (0, 1):
                return ("your function gave a board whose row %d holds %s" % (r, _n3_short(x)),
                        "each dot is her answer, 1 or 0: dot = tap(p[%s], moods[morning])." % index)
    return None


def _n3_too_short(name, test, j):
    """(what, hint) when the first j mornings are the game's and the mirror stops there."""
    k, n, length = len(test["p"]), test["n"], test["test"]
    if name == "initialize_and_test":
        if j < k:
            return ("your function stopped after %s, during the initialization" % _n3_count(j, "morning"),
                    "the initialization shows each product once: for morning in range(len(p)), product = morning.")
        what = ("your test stopped after %s, and the game's has %d: %d to initialize and %s of the hat"
                % (_n3_count(j, "morning"), length, k, _n3_count(n, "morning")))
        if j == length - 1:
            return what, ("the test is one morning short. The initialization takes len(p) mornings and the hat n "
                          "more: test = len(p) + n.")
        return what, "the initialization takes len(p) mornings and the hat n more: test = len(p) + n."
    if j < length:
        return ("your mirror stopped during the test, after %s" % _n3_count(j, "morning"),
                "rows = initialize_and_test(p, moods, next_number, n) plays the initialization and the test, and the "
                "loop over the remaining mornings plays the rest of the season.")
    if j == length:
        return ("your mirror stopped after the %d mornings of the test, and the season has %d" % (length,
                                                                                               test["mornings"]),
                "the loop over the remaining mornings, for morning in range(test, len(moods)), shows the winner on "
                "every morning after the test and adds her answer to its row.")
    return ("your mirror played %s of the %d" % (_n3_count(j, "morning"), test["mornings"]),
            "the remaining mornings run to the last morning of the season: for morning in range(test, len(moods)).")


def _n3_too_long(name, test, events, j):
    """(what, hint) when the first j mornings are the game's and the mirror goes on after them."""
    extra = len(events) - j
    if name == "initialize_and_test":
        what = "your test went on after the %s of the hat, %s more" % (_n3_count(test["n"], "morning"),
                                                                      _n3_count(extra, "morning"))
        if extra == 1:
            return what, ("the test is one morning long. The initialization takes len(p) mornings and the hat n "
                          "more: test = len(p) + n.")
        if len(events) >= test["mornings"]:
            return (what + ", to the end of the season",
                    "the test stops after the n mornings of the hat: for morning in range(len(p), test), with test = "
                    "len(p) + n.")
        return what, "the initialization takes len(p) mornings and the hat n more: test = len(p) + n."
    return ("your mirror asked for her answer %s in a season of %d mornings" % (_n3_count(len(events), "time"),
                                                                                test["mornings"]),
            "one answer per morning: call tap once in each round of the loop.")


def _n3_wrong_morning(name, test, events, j):
    """(what, hint) when the season's j-th answer came from the mood of another morning than morning j."""
    got = events[j][1]
    k, length = len(test["p"]), test["test"]
    upto = test["mornings"]
    if name == "initialize_and_test":
        upto = length
    if name == "estimate_then_exploit" and j == length and got < j:
        if got == 0:
            return ("after the test your mirror started again from morning 0",
                    "the remaining mornings start right after the test: for morning in range(test, len(moods)).")
        return ("after the test your mirror played %s again: its remaining mornings start inside the test"
                % _n3_morning(got),
                "the initialization and the test take the first len(p) + n mornings, so the remaining mornings start "
                "at morning test = len(p) + n: for morning in range(test, len(moods)).")
    if j > 0 and got == events[j - 1][1] and len(events) > upto:
        return ("your mirror asked for her answer on %s twice" % _n3_morning(got),
                "one answer per morning: call tap once in each round of the loop, and append its dot once.")
    if name == "initialize_and_test" or j < length:
        if j == 0:
            return ("your mirror started with the mood of morning %d" % got,
                    "Python counts the mornings from 0: range(len(p)) gives 0, 1, 2 and so on, and moods[0] is the "
                    "mood of the first morning.")
        if j == k and got == 0:
            return ("after the initialization your mirror started again from morning 0",
                    "the mornings of the hat come right after the initialization: for morning in range(len(p), test).")
        if got == length and name == "estimate_then_exploit":
            gap = "mornings %d to %d showed no product" % (j, got - 1)
            if got - 1 == j:
                gap = "morning %d showed no product" % j
            return ("your test stopped after %s of the hat, and your mirror jumped to %s: %s"
                    % (_n3_count(j - k, "morning"), _n3_morning(got), gap),
                    "the test has n mornings of the hat: rows = initialize_and_test(p, moods, next_number, n), with the "
                    "n that the function gets, plays the first test = len(p) + n mornings, and the remaining mornings "
                    "start right after them.")
        if got > j:
            if j < k:
                return ("your initialization skipped %s" % _n3_morning(j),
                        "range(len(p)) gives every morning of the initialization: 0, 1, 2 and so on up to len(p) - 1.")
            return ("your test skipped %s" % _n3_morning(j),
                    "range(len(p), test) gives every morning of the hat: len(p), len(p) + 1 and so on up to test - 1.")
        return ("the %s answer of your test came from the mood of morning %d" % (_n3_ordinal(j + 1), got),
                "each morning has its own mood: dot = tap(p[product], moods[morning]).")
    if got > j:
        return ("your mirror skipped %s" % _n3_morning(j),
                "range(test, len(moods)) starts with morning test, the first morning after the test, and ends with "
                "the last morning of the season.")
    return ("on %s her answer came from the mood of morning %d" % (_n3_morning(j), got),
            "each morning has its own mood: dot = tap(p[winner], moods[morning]).")


def _n3_wrong_init(test, run, j):
    """(what, hint) when on morning j, during the initialization, the mirror showed another product than row j."""
    row = run["events"][j][0]
    if _n3_drawn(run, j) > 0:
        return ("on %s your mirror drew a card from the hat and showed %s. The game's mirror initializes first, with "
                "no card: on that morning it shows %s" % (_n3_morning(j), _n3_product(test, row),
                                                          _n3_product(test, j)),
                "initialize first, with no card: in the first loop product = morning, so morning 0 shows row 0, "
                "morning 1 row 1, and so on. The cards from the hat come after the initialization.")
    return ("on %s your mirror showed %s. Initializing, the game's mirror shows %s" % (_n3_morning(j),
                                                                                      _n3_product(test, row),
                                                                                      _n3_product(test, j)),
            "initialize: each product once, from the top row down. In the first loop product = morning: morning 0 "
            "shows row 0, morning 1 row 1, and so on.")


def _n3_wrong_card(name, fn, test, run, j):
    """(what, hint) when on morning j, a morning of the hat, the mirror showed another product than the game's."""
    events, k, n, length = run["events"], len(test["p"]), test["n"], test["test"]
    row = events[j][0]
    right = int(test["shown"][j])
    what = ("on %s your test showed %s. With the same number from next_number, the game's hat gives %s"
            % (_n3_morning(j), _n3_product(test, row), _n3_product(test, right)))
    if name == "estimate_then_exploit" and len(events) == test["mornings"]:
        # The test stopped early: from morning j on, the mirror shows one product to the end of the season
        same = True
        for t in range(j, len(events)):
            if events[t][0] != row:
                same = False
        if same:
            what = ("your test stopped after %s of the hat, and from %s on your mirror showed %s every morning"
                    % (_n3_count(j - k, "morning"), _n3_morning(j), _n3_product(test, row)))
            hint = ("the test has n mornings of the hat after the initialization: rows = initialize_and_test(p, "
                    "moods, next_number, n), with the n that the function gets, and test = len(p) + n. The winner is "
                    "picked after it.")
            if j == length - 1:
                hint = "the test is one morning short. It has n mornings" + hint[len("the test has n mornings"):]
            return what, hint
    drawn, want = _n3_drawn(run, j), _n3_game_drawn(test, j)
    if drawn == 0:
        return (what + ", and your mirror drew no number from next_number",
                "a card from the hat comes from the mirror's own stream: product = int(next_number() * len(p)), as "
                "Random draws it in notebook 2. next_number is the stream that the function gets.")
    if drawn != want:
        what = ("by %s your mirror had drawn %s from next_number, and the game's mirror %d, one card on each morning "
                "of the hat" % (_n3_morning(j), _n3_count(drawn, "number"), want))
        if drawn > want:
            return what, ("draw exactly one card on each morning of the hat, product = int(next_number() * len(p)), "
                          "and no other number before the test is over. A tie after the test takes the next number.")
        return what, ("draw a new card on every morning of the hat: product = int(next_number() * len(p)) goes "
                      "inside the loop over the mornings of the test.")
    number = _n3_number_at(run, j)
    if number is not None and n > 0 and n != k and row == int(number * n):
        return what, ("multiply by len(p), the number of products: int(next_number() * len(p)) is a row from 0 to "
                      "%d. n counts the mornings of the hat." % (k - 1))
    zero = True
    for t in range(k, j + 1):
        if events[t][0] != 0:
            zero = False
    if zero:
        return what, ("every card is row 0. int(next_number()) is 0 for any number below 1, so multiply first, then "
                      "take int: int(next_number() * len(p)).")
    return what, ("a card from the hat, as Random draws it in notebook 2: product = int(next_number() * len(p)), one "
                  "number on each morning of the hat.")


def _n3_wrong_pick(test, run, j):
    """(what, hint) when on morning j, after the test, the mirror showed another product than the game's."""
    events, k, length, n = run["events"], len(test["p"]), test["test"], test["n"]
    winner, tied = test["winner"], test["tied"]
    row = events[j][0]
    # The mornings after the test on which the mirror went on drawing cards from the hat
    went_on = 0
    t = length
    while t < len(events):
        number = _n3_number_at(run, t)
        if number is None or events[t][0] != int(number * k):
            break
        went_on = went_on + 1
        t = t + 1
    if went_on >= 2 or (went_on == 1 and len(tied) < 2):
        what = ("your test went on after the %s of the hat: from %s on your mirror drew %s more"
                % (_n3_count(n, "morning"), _n3_morning(length), _n3_count(went_on, "card")))
        if went_on == 1:
            return what, ("the test is one morning long. It has len(p) mornings to initialize and n of the hat: "
                          "test = len(p) + n, and on morning test the mirror shows its pick.")
        return what, ("the test has n mornings of the hat after the initialization: rows = initialize_and_test(p, "
                      "moods, next_number, n), with the n that the function gets. On morning test = len(p) + n the "
                      "mirror shows its pick.")
    pick = "the " + test["names"][winner]
    names = []
    for i in tied:
        names.append("the " + test["names"][i])
    if j == length:
        best = 0
        for i in range(len(test["p"])):
            if test["p"][i] > test["p"][best]:
                best = i
        if row == best and run["p_read"]:
            after = "After the test the highest estimate is %s's" % pick
            if len(tied) > 1:
                after = ("After the test %s share the highest estimate, and with the same numbers the game's mirror "
                         "picks %s" % (_n3_and(names), pick))
            return ("on %s your mirror showed %s, the highest probability. %s" % (_n3_morning(j),
                                                                                 _n3_product(test, row), after),
                    "the mirror does not know p: only tap reads it. It picks the highest estimate on its board: "
                    "winner = pick_highest(estimates(rows), next_number).")
        if len(tied) > 1 and row in tied:
            what = ("after the test %s share the highest estimate. On %s your mirror showed %s, and the game's mirror "
                    "picks %s with the next number of its stream" % (_n3_and(names), _n3_morning(j),
                                                                     _n3_product(test, row), pick))
            cards = _n3_drawn(run, length - 1)
            if cards != n:
                return (what + ". Your mirror drew %s from next_number during the test, and the game's hat %d"
                        % (_n3_count(cards, "number"), n),
                        "the cards of the hat and the tie break come from the same stream, in that order: give "
                        "next_number to initialize_and_test, then to pick_highest after it. The tie break takes the "
                        "next number after the cards of the test.")
            if _n3_drawn(run, j) == cards:
                return (what + ". Your mirror drew no number for the tie",
                        "a tie is broken at random with a number from next_number, as in the game, and pick_highest "
                        "does exactly that: winner = pick_highest(estimates(rows), next_number).")
            return what, ("a tie is broken at random with the next number of the stream, as in the game, and "
                          "pick_highest does exactly that: winner = pick_highest(estimates(rows), next_number).")
        return ("on %s, the first morning after the test, your mirror showed %s. The highest estimate after the test "
                "is %s's" % (_n3_morning(j), _n3_product(test, row), pick),
                "the winner is the row with the highest estimate on the board after the test: "
                "winner = pick_highest(estimates(rows), next_number).")
    return ("on %s your mirror switched from %s to %s. The game's mirror shows %s, its pick after the test, on every "
            "remaining morning" % (_n3_morning(j), _n3_product(test, events[j - 1][0]), _n3_product(test, row), pick),
            "pick the winner once, after the test and before the loop over the remaining mornings, and show it every "
            "morning: dot = tap(p[winner], moods[morning]). A pick inside the loop changes the product whenever the "
            "estimates change.")


def _n3_slip(test, run):
    """(what, hint) when the mirror showed, morning by morning, what one of the usual slips shows on this season, or
    None."""
    events = run["events"]
    if len(events) != test["mornings"] or run["p_read"]:
        return None
    rows = ""
    for t in range(len(events)):
        if events[t][1] != t or not isinstance(events[t][0], int):
            return None
        rows = rows + str(events[t][0])
    j = 0
    while j < len(rows) and rows[j] == test["shown"][j]:
        j = j + 1
    if j == len(rows):
        return None
    length, slips = test["test"], test["slips"]
    if slips.get("short") == rows:
        return ("your test is one morning short: on %s your mirror already showed its pick, %s"
                % (_n3_morning(length - 1), _n3_product(test, int(rows[length - 1]))),
                "the test is one morning short. The initialization takes len(p) mornings and the hat n more: rows = "
                "initialize_and_test(p, moods, next_number, n), with the n that the function gets, and test = "
                "len(p) + n.")
    if slips.get("long") == rows:
        return ("your test is one morning long: on %s your mirror drew one more card from the hat"
                % _n3_morning(length),
                "the test is one morning long. It has len(p) mornings to initialize and n of the hat: rows = "
                "initialize_and_test(p, moods, next_number, n), with the n that the function gets, and test = "
                "len(p) + n. On morning test the mirror shows its pick.")
    if slips.get("every") == rows:
        return ("on %s your mirror switched from %s to %s: it picks the highest estimate again every morning"
                % (_n3_morning(j), _n3_product(test, int(rows[j - 1])), _n3_product(test, int(rows[j]))),
                "pick the winner once, after the test and before the loop over the remaining mornings, and show it "
                "every morning: dot = tap(p[winner], moods[morning]). A pick inside the loop changes the product "
                "whenever the estimates change.")
    if slips.get("first") == rows and _n3_drawn(run, length) == _n3_drawn(run, length - 1):
        names = []
        for i in test["tied"]:
            names.append("the " + test["names"][i])
        return ("after the test %s share the highest estimate. Your mirror took the first of them, %s, and with the "
                "same numbers the game's mirror picks %s" % (_n3_and(names), _n3_product(test, int(rows[length])),
                                                              _n3_product(test, test["winner"])),
                "a tie is broken at random with a number from next_number, as in the game, and pick_highest does "
                "exactly that: winner = pick_highest(estimates(rows), next_number).")
    return None


def _n3_explain_mornings(name, fn, run):
    """(what, hint) from the mornings that fn played, read from her answers; None when they are the game's."""
    test, events = run["test"], run["events"]
    upto = test["mornings"]
    if name == "initialize_and_test":
        upto = test["test"]
    expected = []
    for t in range(upto):
        expected.append((int(test["shown"][t]), t))
    for row, morning, swapped, value in events:
        if swapped:
            return ("on %s her answer is the wrong way round: she taps when her mood is below the probability"
                    % _n3_morning(morning),
                    "tap takes the probability first, then the mood: dot = tap(p[%s], moods[morning])."
                    % _n3_index(name, test, morning))
    for row, morning, swapped, value in events:
        if row is None and isinstance(value, int) and not isinstance(value, bool):
            index = _n3_index(name, test, morning)
            return ("on %s her answer came from %d, a row number in the place of a probability"
                    % (_n3_morning(morning), value),
                    "tap takes the probability of the product shown, p[%s], then the mood: dot = tap(p[%s], "
                    "moods[morning])." % (index, index))
        if row is None:
            return ("on %s her answer came from a probability that is not one of p" % _n3_morning(morning),
                    "use p, the probabilities that the function gets: dot = tap(p[%s], moods[morning]). The check also "
                    "tries the other customer and the collections of the challenge round."
                    % _n3_index(name, test, morning))
    if name == "estimate_then_exploit":
        slip = _n3_slip(test, run)
        if slip is not None:
            return slip
    j = 0
    while j < len(events) and j < len(expected) and (events[j][0], events[j][1]) == expected[j]:
        j = j + 1
    if j == len(events) and j == len(expected):
        return None
    if j == len(events):
        return _n3_too_short(name, test, j)
    if j == len(expected):
        return _n3_too_long(name, test, events, j)
    if events[j][1] != j:
        return _n3_wrong_morning(name, test, events, j)
    if j < len(test["p"]):
        return _n3_wrong_init(test, run, j)
    if name == "initialize_and_test" or j < test["test"]:
        return _n3_wrong_card(name, fn, test, run, j)
    return _n3_wrong_pick(test, run, j)


def _n3_rows_problem(name, run):
    """(what, hint) when the mornings are the game's and the board is another: her answers went astray."""
    test, answer, expected = run["test"], run["answer"], run["expected"]
    index = _n3_index(name, test, test["mornings"])
    dots, want, same_lengths = 0, 0, True
    for k in range(len(expected)):
        dots = dots + len(answer[k])
        want = want + len(expected[k])
        if len(answer[k]) != len(expected[k]):
            same_lengths = False
    if dots < want:
        if name == "estimate_then_exploit" and dots == test["test"]:
            return ("your board has the %d dots of the test and none after it" % test["test"],
                    "in the loop over the remaining mornings, add each answer to the winner's row: "
                    "rows[winner].append(dot).")
        return ("your board has %s for %s" % (_n3_count(dots, "dot"), _n3_count(want, "morning")),
                "add each answer to the row of the product shown, once every morning: rows[%s].append(dot)." % index)
    if dots > want:
        return ("your board has %s for %s" % (_n3_count(dots, "dot"), _n3_count(want, "morning")),
                "one dot per morning: append exactly once in each round of the loop.")
    if same_lengths:
        return ("every row has the right number of dots, and some dots are not her answers",
                "add her answer itself: dot = tap(p[%s], moods[morning]), then rows[%s].append(dot)." % (index, index))
    for k in range(len(expected)):
        if len(answer[k]) != len(expected[k]):
            return ("the row of %s has %s, and the game's has %s" % (_n3_product(test, k),
                                                                      _n3_count(len(answer[k]), "dot"),
                                                                      _n3_count(len(expected[k]), "dot")),
                    "her answers are right, and some went into another row. Add each dot to the row of the product "
                    "shown: rows[%s].append(dot)." % index)
    return None


def _n3_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    not_finished = ("⏳ %s is not finished yet: replace each ... (Python calls it Ellipsis) with your code, run its "
                    "cell, then run this check again." % name)
    if _n3_has_gap(fn):
        print(not_finished)
        return
    for other in _N3_NEEDS[name]:
        if _n3_uses(fn, other) and not _n3_works(other):
            print("⏳ %s uses your %s from %s, which does not work yet. Get check_%s(%s) to ✅ first, then run this "
                  "check again." % (name, other, _N3_TASKS[other], other, other))
            return
    failure = _n3_first_failure(name, fn)
    if failure is None:
        print("✅ " + _N3_SUCCESS[name])
        return
    if failure["answer"] is Ellipsis or "ellipsis" in str(failure["error"]).lower():
        print(not_finished)
        return
    test = failure["test"]
    head = "%s, %s" % (test["call"], test["about"])
    if failure["error"] is not None:
        where = ""
        if failure["line"]:
            where = " on the line  %s" % failure["line"]
        print("❌ %s: your function stopped with an error%s. %s" % (head, where, failure["error"]))
        print("   💡 Hint: " + _n3_explain_error(name, failure))
        return
    problem = _n3_shape(name, failure)
    if problem is None:
        problem = _n3_explain_mornings(name, fn, failure)
    if problem is None:
        problem = _n3_rows_problem(name, failure)
    if problem is None:
        problem = ("your board differs from the game's", "compare your code with the steps above the cell, then open "
                                                         "the hints.")
    what, hint = problem
    print("❌ %s: %s." % (head, what))
    print("   💡 Hint: " + hint)


def check_initialize_and_test(fn):
    """Task A: plays initialize_and_test on six seasons (both customers, both collections, n from 0 to 95) with the
    game's stream of each season, and compares each board after the test with the game's, morning by morning."""
    try:
        _n3_check("initialize_and_test", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the three folded cells at the top again, then this cell." % error)


def check_estimate_then_exploit(fn):
    """Task B: plays estimate_then_exploit on eight seasons (both customers, both collections, n from 0 to 95) with
    the game's stream of each season, and compares each board with the game's, morning by morning."""
    try:
        _n3_check("estimate_then_exploit", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the three folded cells at the top again, then this cell." % error)


def play(mirror, *inputs):
    """Plays one season with your mirror and gives its board: play(initialize_and_test, cinderella, moods,
    next_number, 20) is initialize_and_test(cinderella, moods, next_number, 20). When the mirror is not finished yet,
    stops with an error or gives something that is not a board, play prints one calm line and gives None, so the
    cells below still run."""
    import contextlib, io
    try:
        if not callable(mirror):
            print("⏳ play needs your function itself, as in play(initialize_and_test, cinderella, moods, next_number, "
                  "20).")
            return None
        name = getattr(mirror, "__name__", "your mirror")
        if _n3_has_gap(mirror):
            task = ""
            if name in _N3_TASKS:
                task = " (%s)" % _N3_TASKS[name]
            print("⏳ %s is not finished yet%s, so there is no season to show. Finish it, run its cell, then run this "
                  "cell again." % (name, task))
            return None
        for other in _N3_TASKS:
            fn = globals().get(other)
            if other == name or not _n3_uses(mirror, other):
                continue
            if callable(fn) and _n3_has_gap(fn):
                print("⏳ %s uses your %s (%s), which is not finished yet. Finish it, run its cell, then run this cell "
                      "again." % (name, other, _N3_TASKS[other]))
                return None
            if not _n3_works(other):
                print("⏳ %s uses your %s (%s), which does not work yet. Get check_%s(%s) to ✅ first, then run this "
                      "cell again." % (name, other, _N3_TASKS[other], other, other))
                return None
        copies = []
        for x in inputs:
            if isinstance(x, list):
                copies.append(list(x))
            else:
                copies.append(x)
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                answer = mirror(*copies)
        except Exception as error:
            print("❌ %s stopped with an error, %s: %s. Its check above says more." % (name, type(error).__name__,
                                                                                     error))
            return None
        rows = None
        if len(copies) > 0 and isinstance(copies[0], list):
            rows = len(copies[0])
        if not _n3_is_board(answer, rows):
            print("❌ %s gave %s, which is not a board. Its check above says more." % (name, _n3_short(answer)))
            return None
        return answer
    except Exception as error:
        print("❌ play could not run (%s). Run the three folded cells at the top again, then this cell." % error)
        return None


print("✅ The checks of notebook 3 are ready.")

In [ ]:
#@title From notebook 2: the Queen's mirror and Random (run me too) { display-mode: "form" }
# The two mirrors of notebook 2, as in its solutions. Run this cell; open it if you like.

def queen_mirror(p, moods):
    # The Queen's mirror knows every probability: every morning it shows the product with the highest one.
    best = 0
    for i in range(len(p)):
        if p[i] > p[best]:      # product i beats the best so far
            best = i
    rows = []
    for i in range(len(p)):
        rows.append([])         # one empty row per product
    for morning in range(len(moods)):
        dot = tap(p[best], moods[morning])    # 1 if she taps the best product this morning, else 0
        rows[best].append(dot)                # one more dot in its row
    return rows


def random_mirror(p, moods, next_number):
    # Random draws a card from the hat each morning: every product has the same chance.
    rows = []
    for i in range(len(p)):
        rows.append([])         # one empty row per product
    for morning in range(len(moods)):
        card = int(next_number() * len(p))    # a card from the hat: a row from 0 to len(p) - 1
        dot = tap(p[card], moods[morning])    # her answer this morning: 1 or 0
        rows[card].append(dot)                # one more dot in the card's row
    return rows


print("✅ queen_mirror and random_mirror from notebook 2 are ready.")

## The strategy in four parts

The game told estimate then exploit in four parts:

1. **Initialize**: show each product once.
2. **Test**: draw a card from the hat for n mornings.
3. **Highest estimate**: pick the product with the highest estimate.
4. **Exploit**: show it on every remaining morning.

The test is Random from notebook 2, for n mornings. The estimates it leaves on the board decide the rest of the
season. Here are the two customers, with the probability that each of them taps each product, and the moods of
the race season, the season of every comparison in the game.

In [ ]:
# The probability that each customer taps each product, in board order (invented for the game).
# Only tap reads them: the mirror itself sees only the dots of its board.
cinderella = [0.6, 0.2, 0.2, 0.4, 0.2]
rapunzel = [0.4, 0.8, 0.2, 0.2, 0.2]

# The race season of the game: her mood on each of its mornings
moods = season_moods(RACE)

for i in range(len(NAMES)):
    print("row", i, "   Cinderella", cinderella[i], "   Rapunzel", rapunzel[i], "   ", EMOJI[i], NAMES[i])

## The hat

On each morning of the test the mirror draws a card from the hat, as `random_mirror` does in notebook 2:
`int(next_number() * len(p))` turns a number between 0 and 1 into a row from 0 to `len(p) - 1`, each row with the
same chance. Estimate then exploit draws from its own stream, `strategy_numbers(RACE, "estimate then exploit")` in the race season,
as in the game. Python counts the mornings from 0, so the initialization takes mornings 0 to 4 and the test
starts on morning 5. The cell draws the cards of mornings 5 to 10.

In [ ]:
# The first cards of the test in the race season: one number of the mirror's own stream per morning of the hat
next_number = strategy_numbers(RACE, "estimate then exploit")
for morning in range(5, 11):
    number = next_number()
    card = int(number * len(cinderella))      # a number between 0 and 1 becomes a row from 0 to 4
    print("morning", morning, "   number", round(number, 3), "   card: row", card, EMOJI[card], NAMES[card])

## 🎯 Task A. Initialize, then the test

Write `initialize_and_test(p, moods, next_number, n)`. It plays the first mornings of the season and returns the
board after the test. The board is started for you, and so are her answer and its dot on each morning, as in
notebook 2. You fill in three lines:

1. **Initialize**: on morning 0 the product of row 0, on morning 1 row 1, and so on to row `len(p) - 1`.
2. **The length of the test**: `test` counts the mornings of the initialization and of the hat, `len(p) + n`, so
   25 mornings for n = 20. `range(len(p), test)` then gives the mornings of the hat.
3. **A card from the hat** on each of those mornings, as in the cell above.

On the race season, `initialize_and_test(cinderella, moods, strategy_numbers(RACE, "estimate then exploit"), 20)` gives the
game's board after the test.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# Initializing, morning m shows row m: the product is the morning itself.
# The initialization takes len(p) mornings and the hat n more: test = len(p) + n.
# A card from the hat is int(next_number() * len(p)), as in random_mirror of notebook 2.

In [ ]:
#@title 💡 Hint 2 for task A: the shape of the code (double-click to open) { display-mode: "form" }
#     for morning in range(len(p)):
#         product = morning
#
#     test = len(p) + n
#     for morning in range(len(p), test):
#         product = int(next_number() * len(p))

In [ ]:
# ✅ SOLUTION, task A: initialize, then the test
def initialize_and_test(p, moods, next_number, n):
    # p: the probability of a tap for each product, in board order. Only tap reads it.
    # moods: her mood on each morning of the season. next_number: the mirror's own stream of numbers.
    # n: the mornings of the hat. Returns the board after the test.
    rows = []
    for i in range(len(p)):
        rows.append([])                         # one empty row per product

    # 1. Initialize: each product once, from the top row down
    for morning in range(len(p)):
        product = morning                       # morning 0 shows row 0, morning 1 row 1, and so on
        dot = tap(p[product], moods[morning])   # her answer: 1 for a tap, 0 for no tap
        rows[product].append(dot)               # one more dot in that product's row

    # 2. Test: a card from the hat on each of the next n mornings
    test = len(p) + n                           # len(p) mornings to initialize, then n of the hat
    for morning in range(len(p), test):
        product = int(next_number() * len(p))   # a card from the hat: a row from 0 to len(p) - 1
        dot = tap(p[product], moods[morning])   # her answer: 1 for a tap, 0 for no tap
        rows[product].append(dot)               # one more dot in that product's row
    return rows

In [ ]:
check_initialize_and_test(initialize_and_test)

## The board after the test

`play(initialize_and_test, cinderella, moods, next_number, 20)` runs `initialize_and_test(cinderella, moods,
next_number, 20)` and gives its board. While the function is not finished, it prints one calm line and gives
`None`.

In [ ]:
# Initialize and test on the race season at Cinderella's, with the game's n = 20
next_number = strategy_numbers(RACE, "estimate then exploit")      # the mirror's own stream, as in the game
test_rows = play(initialize_and_test, cinderella, moods, next_number, 20)
show_board(test_rows)

**Reading the result.** The game's board after the test. The hat gave the mattress 7 of its 20 cards and the vacuum 2, so
with the initialization the mattress has 8 dots and the vacuum 3. The vacuum got 2 taps in 3 showings, an estimate of 0.67;
the mattress 3 in 8, 0.38; the satnav 1 in 5, 0.2; the ladder and the blower no tap at all.

## The highest estimate

After the test, the mirror picks the product with the highest estimate. The kit's `pick_highest(values,
next_number)` gives its row. When several rows share the highest value, it picks one of them at random with a
number from `next_number`, exactly as the game does, and it draws a number only when there is a tie. The cards and
the tie come from the same stream, in that order: a tie takes the next number after the last card.

With n = 20 the test ends with one product ahead. With n = 6 it ends in a tie. The cell plays both tests on the
race season, each with a fresh stream, and picks after each.

In [ ]:
# The highest estimate after the test of the race season, with n = 20 and with n = 6
for n in [20, 6]:
    next_number = strategy_numbers(RACE, "estimate then exploit")    # a fresh stream: first the cards, then a tie
    rows = play(initialize_and_test, cinderella, moods, next_number, n)
    if rows != None:
        show_board(rows)
        print("n =", n, ": the highest estimate is row", pick_highest(estimates(rows), next_number))
        print()

**Reading the result.** With n = 20 the vacuum leads alone, 2/3 against the mattress's 3/8: row 0, and no number drawn. With
n = 6 the vacuum and the mattress share the highest estimate, 0.5: `pick_highest` drew number 7 of the stream, the one
after the 6 cards, and chose row 3, the mattress. The same stream always makes the same choice, so your mirror picks
as the game's does.

## 🎯 Task B. Estimate then exploit

Write `estimate_then_exploit(p, moods, next_number, n)`. It plays a whole season and returns its board, in three
steps:

1. **Initialize and test**: your `initialize_and_test` from task A plays the first `len(p) + n` mornings.
2. **Highest estimate**: `pick_highest(estimates(rows), next_number)` gives the winner. Pick it once, after the
   test.
3. **Exploit**: the winner on every remaining morning. `range(test, len(moods))` counts the remaining mornings:
   `range(25, 100)` gives 25, 26, 27 and so on up to 99.

Give `next_number` to both functions: the cards of the test come first, then a number for a tie, as in the game.
With n = 20 and the stream `strategy_numbers(RACE, "estimate then exploit")`, your mirror plays the game's race season, morning by
morning.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# Step 1: initialize_and_test(p, moods, next_number, n) gives the board after the test.
# Step 2: estimates(rows) gives the estimate of every row, and pick_highest(values, next_number) the row with
# the highest one. Pick once, after the test and before the loop.
# Step 3: each remaining morning, her answer to the winner is tap(p[winner], moods[morning]), and append puts
# it at the end of the winner's row.

In [ ]:
#@title 💡 Hint 2 for task B: the shape of the code (double-click to open) { display-mode: "form" }
#     rows = initialize_and_test(p, moods, next_number, n)
#     test = len(p) + n
#     winner = pick_highest(estimates(rows), next_number)
#
#     for morning in range(test, len(moods)):
#         dot = tap(p[winner], moods[morning])
#         rows[winner].append(dot)

In [ ]:
# ✅ SOLUTION, task B: estimate then exploit
def estimate_then_exploit(p, moods, next_number, n):
    # p, moods, next_number and n as in task A. Returns the board of the whole season.

    # 1. Initialize and test: each product once, then n mornings of the hat
    rows = initialize_and_test(p, moods, next_number, n)    # the board after the test, from task A
    test = len(p) + n                           # the mornings of the initialization and the hat

    # 2. The highest estimate, picked once
    winner = pick_highest(estimates(rows), next_number)

    # 3. Exploit: the winner on every remaining morning
    for morning in range(test, len(moods)):
        dot = tap(p[winner], moods[morning])    # her answer to the winner: 1 or 0
        rows[winner].append(dot)                # one more dot in the winner's row
    return rows

In [ ]:
check_estimate_then_exploit(estimate_then_exploit)

## Cinderella's season

💭 **Think first.** With n = 20 the initialization and the test take 25 mornings, and the hat gives each product
4 of its 20 cards on average. If the mirror then picks the vacuum, she sees it on the 75 remaining mornings and taps
it with probability 0.6. About how many taps do you expect in the season? And from the Queen's mirror and Random,
on the same mornings?

In [ ]:
# Estimate then exploit at Cinderella's on the race season, beside the Queen's mirror and Random
n = 20
ete_rows = play(estimate_then_exploit, cinderella, moods, strategy_numbers(RACE, "estimate then exploit"), n)
show_board(ete_rows)

queen_rows = play(queen_mirror, cinderella, moods)
random_rows = play(random_mirror, cinderella, moods, strategy_numbers(RACE, "random"))
print("Estimate then exploit, n =", n, ":", rate(ete_rows))
print("The Queen's mirror:", rate(queen_rows))
print("Random:", rate(random_rows))

**Reading the result.** In the 25 mornings of the initialization and the test she tapped 6 times. Then the vacuum,
2/3 against the mattress's 3/8, on the 75 remaining mornings: 41 taps. In all 47 taps, a rate of 0.47, between Random's 0.31
and the Queen's mirror's 0.6: the game's numbers. The board shows the first 30 dots of the vacuum's row and counts the
other 48.

With n = 10 the test ends with the mattress ahead, 3/5 against the vacuum's 1/2, and the season brings 35 taps. Change n in
the cell to see it.

## The challenge round

In the game's challenge round you chose n for two new collections, built on purpose:

- **Ready for the ball**, at Cinderella's. Cinderella is getting ready for the ball. Her mirror has five new products for the evening.
- **Long hair**, at Rapunzel's. Rapunzel's hair is very, very long. Her mirror has five new products for it.

Here are their probabilities, in the game's order. Only tap reads them: the mirror sees the dots of its board, as
always.

In [ ]:
# The two collections of the challenge round, rows in the game's order (invented for the game)
# Ready for the ball, at Cinderella's: silk gloves, tiara, evening bag, ball gown, lipstick
ball = [0.6, 0.9, 0.6, 0.95, 0.6]
ball_emoji = ['🧤', '👑', '👜', '👗', '💄']
# Long hair, at Rapunzel's: scissors, cap, ribbon, razor, haircut voucher
hair = [0.02, 0.02, 0.95, 0.02, 0.02]
hair_emoji = ['✂️', '🧢', '🎀', '🪒', '💇']

for i in range(len(ball)):
    print("row", i, "   ", ball_emoji[i], ball[i], "   ", hair_emoji[i], hair[i])

💭 **Think first.** At the ball, two products are almost equally good, the tiara and the ball gown, and three decoys are
not far behind. For the long hair, the ribbon stands far above four products she almost never taps. For each
collection, which n would you choose: 0, 3, 10, 25, 50 or 95?

`one_season(p, symbols, n, seed)` plays one season: estimate then exploit with your n, the Queen's mirror and
Random, all three on the same moods. It shows the board of estimate then exploit with the collection's pictures,
then the three rates.

In [ ]:
# Given: one season of a collection, for the three mirrors
def one_season(p, symbols, n, seed):
    # Plays the season with this seed: estimate then exploit with n, the Queen's mirror and Random, all three on
    # the same moods. Shows the board of estimate then exploit and prints the three rates.
    moods = season_moods(seed)
    rows = play(estimate_then_exploit, p, moods, strategy_numbers(seed, "estimate then exploit"), n)
    if rows == None:
        return                  # estimate_then_exploit is not ready yet: play has said why
    show_board(rows, symbols=symbols)
    print("Estimate then exploit, n =", n, ":", rate(rows))
    print("The Queen's mirror:", rate(play(queen_mirror, p, moods)))
    print("Random:", rate(play(random_mirror, p, moods, strategy_numbers(seed, "random"))))

In [ ]:
# One season of each collection with your n: change n (or seed) and run the cell again
n = 0
seed = 2
one_season(ball, ball_emoji, n, seed)
print()
one_season(hair, hair_emoji, n, seed)

**Reading the result.** With n = 0 there is no test: each product gets one showing, then the mirror picks. At
the ball Cinderella tapped three of them, so the tiara, the evening bag and the ball gown shared the highest estimate, 1, and the tie went to the evening bag, a decoy at 0.6: a
rate of 0.56, against 0.92 for the Queen's mirror. At Rapunzel's the ribbon won at once: 0.88, against the Queen's mirror's 0.92.
One season is one roll of luck: change n, or the seed, and run the cell again.

## Many seasons

`average_rates(p, n_values, first_seed, seasons)` plays the three mirrors for many seasons, one season per seed,
with every n in the list `n_values`, and prints the rate of all those mornings. In each season every mirror meets
the same moods. Each cell below takes a few seconds.

In [ ]:
# Given: the taps of a board, and the three mirrors over many seasons
def count_taps(rows):
    # All the 1s on the board, added up
    taps = 0
    for row in rows:
        for dot in row:
            taps = taps + dot
    return taps


def average_rates(p, n_values, first_seed, seasons):
    # Plays the Queen's mirror, Random, and estimate then exploit with every n in n_values, for many seasons, one
    # season per seed, and prints the rate of all those mornings. In each season every mirror meets the same moods.
    queen_taps = 0
    random_taps = 0
    ete_taps = []
    for n in n_values:
        ete_taps.append(0)          # one count of taps for each n
    for season in range(seasons):
        seed = first_seed + season
        moods = season_moods(seed)
        queen_taps = queen_taps + count_taps(play(queen_mirror, p, moods))
        random_taps = random_taps + count_taps(play(random_mirror, p, moods, strategy_numbers(seed, "random")))
        for k in range(len(n_values)):
            rows = play(estimate_then_exploit, p, moods, strategy_numbers(seed, "estimate then exploit"), n_values[k])
            if rows == None:
                return              # estimate_then_exploit is not ready yet: play has said why
            ete_taps[k] = ete_taps[k] + count_taps(rows)
    mornings = seasons * SEASON
    print("Seasons:", seasons, "   first seed:", first_seed, "   mornings in all:", mornings)
    print("The Queen's mirror:", queen_taps / mornings)
    print("Random:", random_taps / mornings)
    for k in range(len(n_values)):
        print("Estimate then exploit, n =", n_values[k], ":", ete_taps[k] / mornings)

In [ ]:
# 1,000 seasons of Ready for the ball, at Cinderella's: seeds 1 to 1000, and the n of the game's challenge round
average_rates(ball, [0, 3, 10, 25, 50, 95], 1, 1000)

**Reading the result.** The Queen's mirror: 0.95031, about 0.95, the ball gown every morning. Random: 0.72993, about the average of
the five probabilities. Estimate then exploit does best with n = 25, about 0.84. With n = 0 it gets about 0.78: after
one showing each, 986 of these 1,000 seasons had a tie for the highest estimate, and the mirror found the ball gown only 275
times. With n = 95 the test fills the whole season, 5 mornings to initialize and 95 of the hat, and the rate is
Random's, about 0.73.

In [ ]:
# 1,000 seasons of Long hair, at Rapunzel's: the same seeds
average_rates(hair, [0, 3, 10, 25, 50, 95], 1, 1000)

**Reading the result.** The ribbon usually stands out after one showing: n = 0 found it in 908 of the 1,000 seasons, and
n = 3 does best, about 0.85, close to the Queen's mirror's 0.95. Four times in five, a card from the hat shows a
product she taps 2 times in 100: with n = 50 the initialization and the test take 55 of the 100 mornings, about
0.54, and with n = 95 they take all of them, about 0.21, Random's rate.

## Choosing n is hard

A large n wastes mornings on exploration: for the long hair, n = 95 spends the whole season on the hat. A small n
risks not exploring enough: at the ball, n = 0 usually misses the ball gown. The best n depends on the probabilities, and the mirror does not know them.

## Back to the game

That is notebook 3. Section 4 of the game, **Epsilon-greedy**, shows a strategy that never stops exploring.
Notebook 4 comes after it.